# Lista 12 Zadanie 1

In [71]:
import pandas as pd
import numpy as np
import itertools
from collections import defaultdict
import random
import time

## Podpunkt 1

In [4]:
df = pd.read_csv('market.csv', sep=';')

In [5]:
df

,Bread,Honey,Bacon,Toothpaste,Banana,Apple,Hazelnut,Cheese,Meat,Carrot,...,Milk,Butter,ShavingFoam,Salt,Flour,HeavyCream,Egg,Olive,Shampoo,Sugar
0,1,0,1,0,1,1,1,0,0,1,...,0,0,0,0,0,1,1,0,0,1
1,1,1,1,0,1,1,1,0,0,0,...,1,1,0,0,1,0,0,1,1,0
2,0,1,1,1,1,1,1,1,1,0,...,1,0,1,1,1,1,1,0,0,1
3,1,1,0,1,0,1,0,0,0,0,...,1,0,0,0,1,0,1,1,1,0
4,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
459,0,1,1,0,0,1,1,1,1,1,...,1,1,0,0,1,1,1,1,1,0
460,0,0,1,0,0,0,1,0,1,0,...,0,0,1,0,0,0,1,0,0,1
461,0,0,0,0,0,1,0,1,1,0,...,1,0,0,0,0,0,1,0,0,0
462,1,0,0,1,1,0,1,1,0,1,...,1,0,0,0,1,0,1,1,0,1


In [48]:
cols = df.columns

for col1 in range(0, len(cols)):
    for col2 in range(col1 + 1, len(cols)):
        res = df[df[cols[col1]] == 1][cols[col2]].value_counts()
        support = res[1] / len(df) * 100
        confidence = res[1] / (res[0] + res[1]) * 100
        if support > 20 and confidence > 50:
            print(f'\n{cols[col1]} vs {cols[col2]}')
            print(f'Support {support:.2f}%')
            print(f'Confidence: {confidence:.2f}%')



Bread vs Banana
Support 21.12%
Confidence: 51.85%

Bread vs Cheese
Support 21.55%
Confidence: 52.91%

Bread vs Salt
Support 20.69%
Confidence: 50.79%

Honey vs Banana
Support 21.34%
Confidence: 51.30%

Honey vs Cheese
Support 21.77%
Confidence: 52.33%

Honey vs Egg
Support 20.91%
Confidence: 50.26%

Bacon vs Banana
Support 24.14%
Confidence: 56.00%

Bacon vs Hazelnut
Support 22.20%
Confidence: 51.50%

Bacon vs Cheese
Support 22.41%
Confidence: 52.00%

Toothpaste vs Banana
Support 20.26%
Confidence: 52.81%

Toothpaste vs Hazelnut
Support 20.26%
Confidence: 52.81%

Toothpaste vs Carrot
Support 20.47%
Confidence: 53.37%

Milk vs HeavyCream
Support 20.69%
Confidence: 55.81%

Salt vs HeavyCream
Support 20.69%
Confidence: 51.89%


Czesto czyli te dwa przedmioty wystepuja conajmniej w 20% transakcji i czesciej sa kupowane razem niz osobno.

Najlepiej ustawic taki prog, zeby nie byl zbyt wysoki ani zbyt niski. Zbyt wysoki tracimy potencjalnie ciekawe reguly, a zbyt niski to zwiekszony czas obliczen.

## Podpunkt 2

In [53]:
def get_support(transactions, candidates, min_support):
    counts = defaultdict(int)
    for transaction in transactions:
        for candidate in candidates:
            if candidate.issubset(transaction):
                counts[candidate] += 1

    n = len(transactions)
    return {item: count/n for item, count in counts.items() if (count/n) >= min_support}


In [54]:
def run_apriori(transactions, min_support):
    all_frequent_itemsets = {}

    items = set(item for trans in transactions for item in trans)
    c1 = [frozenset([item]) for item in items]

    l1 = get_support(transactions, c1, min_support)
    all_frequent_itemsets.update(l1)

    current_l = list(l1.keys())
    k = 2

    while len(current_l) > 0:
        candidates = set()
        for i in range(len(current_l)):
            for j in range(i + 1, len(current_l)):
                union = current_l[i] | current_l[j]
                if len(union) == k:
                    candidates.add(union)

        lk = get_support(transactions, candidates, min_support)
        all_frequent_itemsets.update(lk)

        current_l = list(lk.keys())
        k += 1

    return all_frequent_itemsets

## Podpunkt 3 i 4

In [55]:
def get_rules(frequent_itemsets, min_confidence):
    rules = []
    for itemset, support in frequent_itemsets.items():
        if len(itemset) > 1:
            for i in range(1, len(itemset)):
                for antecedent in itertools.combinations(itemset, i):
                    ant = frozenset(antecedent)
                    conseq = itemset - ant

                    conf = support / frequent_itemsets[ant]
                    lift = conf / frequent_itemsets[conseq]

                    if conf >= min_confidence:
                        rules.append((list(ant), list(conseq), support, conf, lift))
    return rules


Support ustawiam na 0.1, zeby nie bylo za duzo wynikow.

Confidence ustawiam na 0.5. Jest to dobry balans miedzy zbyt pewnymi polaczeniami, a zasmiecaniem wynikow.

In [69]:
transactions = [set(row.index[row == 1]) for _, row in df.iterrows()]
freq_sets = run_apriori(transactions, min_support=0.1)
final_rules = get_rules(freq_sets, min_confidence=0.5)

In [70]:
for a, b, sup, conf, lift in final_rules:
    print(f"{a} -> {b} | Sup: {sup:.2f}, Conf: {conf:.2f}, Lift: {lift:.2f}")

['Carrot'] -> ['Banana'] | Sup: 0.22, Conf: 0.53, Lift: 1.19
['Bacon'] -> ['Hazelnut'] | Sup: 0.22, Conf: 0.52, Lift: 1.23
['Hazelnut'] -> ['Bacon'] | Sup: 0.22, Conf: 0.53, Lift: 1.23
['Sugar'] -> ['Bacon'] | Sup: 0.19, Conf: 0.51, Lift: 1.19
['Bread'] -> ['Banana'] | Sup: 0.21, Conf: 0.52, Lift: 1.16
['Egg'] -> ['Bacon'] | Sup: 0.20, Conf: 0.50, Lift: 1.17
['Hazelnut'] -> ['Banana'] | Sup: 0.22, Conf: 0.53, Lift: 1.18
['Egg'] -> ['Banana'] | Sup: 0.21, Conf: 0.53, Lift: 1.18
['Bacon'] -> ['Banana'] | Sup: 0.24, Conf: 0.56, Lift: 1.25
['Banana'] -> ['Bacon'] | Sup: 0.24, Conf: 0.54, Lift: 1.25
['Apple'] -> ['Banana'] | Sup: 0.22, Conf: 0.54, Lift: 1.21
['Egg'] -> ['Carrot'] | Sup: 0.20, Conf: 0.50, Lift: 1.21
['HeavyCream'] -> ['Hazelnut'] | Sup: 0.21, Conf: 0.50, Lift: 1.20
['Shampoo'] -> ['Honey'] | Sup: 0.19, Conf: 0.51, Lift: 1.23
['Honey'] -> ['Banana'] | Sup: 0.21, Conf: 0.51, Lift: 1.14
['Butter'] -> ['Bread'] | Sup: 0.20, Conf: 0.52, Lift: 1.28
['Olive'] -> ['Banana'] | Sup: 0

Aby wybrac najbardziej wartosciowe, warto spojrzec na lift. Mowi o tym jak bardzo kupione produkty maja wplyw na wybor nastepnego.

Najpierw bym odfiltrowal wszystkie z malym lift np. 1.2, a potem sortowal po confidence.

## Podpunkt 5

In [73]:
def expand_dataset(original_transactions, target_n=1000000):
    expanded = []
    all_items = list(set().union(*original_transactions))

    for _ in range(target_n):
        base = set(random.choice(original_transactions))

        rand_val = random.random()
        if rand_val < 0.3:
            if len(base) > 1:
                base.remove(random.choice(list(base)))
        elif rand_val < 0.6:
            base.add(random.choice(all_items))

        expanded.append(frozenset(base))
    return expanded

In [74]:
big_data = expand_dataset(transactions)

In [85]:
start = time.time()
freq_sets = run_apriori(big_data, min_support=0.1)
print(f'Apriori zajelo: {time.time() - start:.2f} sekund')

Apriori zajelo: 115.62 sekund


In [87]:
start = time.time()
final_rules = get_rules(freq_sets, min_confidence=0.5)
print(f'Reguly zajely: {time.time() - start:.2f} sekund')

Reguly zajely: 0.00 sekund


## Podpunkt 6

In [80]:
def add_thousand_new_products(transactions, n_new=1000):
    new_items = [f"New_item_{i}" for i in range(n_new)]
    modified_transactions = []

    for trans in transactions:
        items = list(trans)
        for i in range(len(items)):
            if random.random() < 0.2:
                items[i] = random.choice(new_items)
        modified_transactions.append(frozenset(items))

    return modified_transactions

In [90]:
bigger_data = expand_dataset(add_thousand_new_products(transactions))

In [91]:
start = time.time()
freq_sets = run_apriori(bigger_data, min_support=0.1)
print(f'Apriori zajelo: {time.time() - start:.2f} sekund')

Apriori zajelo: 64.51 sekund


Wiecej danych zajelo mniej czasu. Pewnie dlatego, ze nie przekraczaja progu supportu